# Census Block Level Roll-Up — Illinois (IL)

This notebook implements **Phase 1, Step 1.3** of the Broadband Pipeline Plan.  
It takes the raw FCC location-level CSV files for Illinois and aggregates them to the **census block level**, producing a clean dataset ready for PostgreSQL loading.

**Roll-up logic (per the plan):**
```
For each unique combination of (block_geoid, provider_id, holding_company, technology):
    → Take the MAX of max_advertised_download_speed
    → Take the MAX of max_advertised_upload_speed
    → Flag whether residential, business, or both are served
    → Take the MAX of low_latency
```

## 1. Import Libraries

In [1]:
import os
import pandas as pd

## 2. Load All IL CSV Files

In [2]:
### Reading all the IL CSV files into a single DataFrame
path = '../data/IL/'
il_data = pd.DataFrame()
for file in os.listdir(path):
    if file.endswith('.csv'):
        df = pd.read_csv(os.path.join(path, file))
        print(f"Loaded {file}: {df.shape[0]:,} rows")
        il_data = pd.concat([il_data, df], ignore_index=True)

print(f"\nTotal IL records: {il_data.shape[0]:,}")
il_data.head()

Loaded bdc_17_FibertothePremises_fixed_broadband_J25_03mar2026.csv: 2,374,355 rows
Loaded bdc_17_Cable_fixed_broadband_J25_03mar2026.csv: 4,056,296 rows
Loaded bdc_17_Copper_fixed_broadband_J25_03mar2026.csv: 2,188,064 rows

Total IL records: 8,618,715


,frn,provider_id,brand_name,location_id,technology,max_advertised_download_speed,max_advertised_upload_speed,low_latency,business_residential_code,state_usps,block_geoid,h3_res8_id
0,3710597,390138,Gridley Telephone Co.,1412511090,50,1000,1000,1,X,IL,171130057004025,8826629197fffff
1,7494776,130335,Fidium Fiber,1022019182,50,2000,2000,1,R,IL,170219584003012,8826448d55fffff
2,7494776,130335,Fidium Fiber,1123983754,50,2000,2000,1,R,IL,171359577002012,882644723bfffff
3,7494776,130335,Fidium Fiber,1130271974,50,2000,2000,1,R,IL,170290009003009,8826630e37fffff
4,7494776,130335,Fidium Fiber,1130277016,50,2000,2000,1,R,IL,170290012003008,882663735bfffff


## 3. Basic Data Checks

In [3]:
### Data types and memory usage
print(il_data.info())
print("\n--- Missing Values ---")
print(il_data.isnull().sum())
print("\n--- Unique technology codes ---")
print(il_data['technology'].unique())
print("\n--- Unique providers ---")
print(f"{il_data['provider_id'].nunique()} unique provider IDs")
print(f"{il_data['brand_name'].nunique()} unique brand names")
print("\n--- Business/Residential split ---")
print(il_data['business_residential_code'].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 8618715 entries, 0 to 8618714
Data columns (total 12 columns):
 #   Column                         Dtype
---  ------                         -----
 0   frn                            int64
 1   provider_id                    int64
 2   brand_name                     str  
 3   location_id                    int64
 4   technology                     int64
 5   max_advertised_download_speed  int64
 6   max_advertised_upload_speed    int64
 7   low_latency                    int64
 8   business_residential_code      str  
 9   state_usps                     str  
 10  block_geoid                    int64
 11  h3_res8_id                     str  
dtypes: int64(8), str(4)
memory usage: 789.1 MB
None

--- Missing Values ---
frn                              0
provider_id                      0
brand_name                       0
location_id                      0
technology                       0
max_advertised_download_speed    0
max_advertised_upload_s

## 4. Provider Mapping — Merge with Master Provider List

In [4]:
### Reading providers master sheet
providers_master = pd.read_csv('../output/final_master_set.csv')
providers_master.rename(columns={'Provider ID': 'provider_id', 'FRN': 'frn'}, inplace=True)
print(f"Master provider records: {providers_master.shape[0]:,}")
providers_master.head()

Master provider records: 2,883


,provider_id,frn,holding_company
0,240058,12609,Mid-Hudson Cablevision
1,240068,12781,"Neu Ventures, Inc."
2,131087,13722,"Rainbow Telecommunications Association, Inc."
3,240058,14936,Mid-Hudson Cablevision
4,131060,17640,"TPC ACC Acquisition, LLC"


In [5]:
### Merging IL data with providers master to get holding_company
il_data = pd.merge(il_data, providers_master, on=['provider_id', 'frn'], how='left')

### Check for unmatched providers (NaN holding_company)
unmatched = il_data[il_data['holding_company'].isna()]['brand_name'].unique()
if len(unmatched) > 0:
    print(f"WARNING: {len(unmatched)} brand(s) did not match the master list:")
    print(unmatched)
else:
    print("All providers matched successfully.")

print(f"\nMerged shape: {il_data.shape}")

All providers matched successfully.

Merged shape: (8618715, 13)


## 5. Technology Code Reference (Step 1.2)

| Code | Technology |
|------|------------|
| 10   | Copper Wire (DSL) |
| 40   | Cable |
| 50   | Fiber to the Premises (FTTP) |
| 0    | Other |

In [6]:
### Technology code to human-readable name mapping
TECHNOLOGY_MAP = {
    10: 'Copper Wire (DSL)',
    40: 'Cable',
    50: 'Fiber to the Premises (FTTP)'
}

### Verify all technology codes in the data are covered by the map
codes_in_data = set(il_data['technology'].unique())
codes_in_map = set(TECHNOLOGY_MAP.keys())
unmapped = codes_in_data - codes_in_map

if unmapped:
    print(f"WARNING: Unmapped technology codes found: {unmapped}")
else:
    print(f"All technology codes covered. Codes present in data: {sorted(codes_in_data)}")

All technology codes covered. Codes present in data: [np.int64(10), np.int64(40), np.int64(50)]


## 6. Data Cleaning — block_geoid Preparation

The `block_geoid` is a 15-digit Census Block FIPS code. We need to:
1. Drop rows with null or malformed `block_geoid`
2. Zero-pad to 15 digits (leading zeros are stripped when stored as integers)
3. Derive `state_fips` (first 2 digits), `county_fips` (first 5), and `tract_fips` (first 11)

In [7]:
### Check for null block_geoid values
null_geoid_count = il_data['block_geoid'].isna().sum()
print(f"Null block_geoid rows: {null_geoid_count:,}")

### Drop rows with null block_geoid
if null_geoid_count > 0:
    il_data = il_data.dropna(subset=['block_geoid'])
    print(f"Dropped {null_geoid_count:,} rows. Remaining: {il_data.shape[0]:,}")

### Zero-pad block_geoid to 15 digits
il_data['block_geoid'] = il_data['block_geoid'].astype(int).astype(str).str.zfill(15)

### Validate: all block_geoid should be exactly 15 digits
invalid_geoid = il_data[il_data['block_geoid'].str.len() != 15]
print(f"Invalid block_geoid (not 15 digits): {invalid_geoid.shape[0]:,}")

### Derive FIPS hierarchy columns
il_data['state_fips'] = il_data['block_geoid'].str[:2]
il_data['county_fips'] = il_data['block_geoid'].str[:5]
il_data['tract_fips'] = il_data['block_geoid'].str[:11]

print(f"\nState FIPS values: {il_data['state_fips'].unique()}")
print(f"Unique counties: {il_data['county_fips'].nunique()}")
print(f"Unique tracts: {il_data['tract_fips'].nunique()}")
print(f"Unique census blocks: {il_data['block_geoid'].nunique()}")

Null block_geoid rows: 0
Invalid block_geoid (not 15 digits): 0

State FIPS values: <StringArray>
['17']
Length: 1, dtype: str
Unique counties: 102
Unique tracts: 3262
Unique census blocks: 273943


## 7. Census Block Level Roll-Up (Step 1.3)

Aggregate from location-level rows to census block level:
- Group by `(block_geoid, provider_id, holding_company, technology)`
- MAX of download and upload speeds
- Flag residential/business service
- MAX of low_latency

In [8]:
### Roll up to census block level
block_level = il_data.groupby(
    ['block_geoid', 'provider_id', 'holding_company', 'technology']
).agg(
    max_download=('max_advertised_download_speed', 'max'),
    max_upload=('max_advertised_upload_speed', 'max'),
    serves_residential=('business_residential_code', lambda x: any(v in ['R', 'X'] for v in x)),
    serves_business=('business_residential_code', lambda x: any(v in ['B', 'X'] for v in x)),
    low_latency=('low_latency', 'max'),
    state_fips=('state_fips', 'first'),
    county_fips=('county_fips', 'first'),
    tract_fips=('tract_fips', 'first')
).reset_index()

print(f"Location-level rows: {il_data.shape[0]:,}")
print(f"Block-level rows:    {block_level.shape[0]:,}")
print(f"Reduction:           {il_data.shape[0] / block_level.shape[0]:.1f}x")

Location-level rows: 8,618,715
Block-level rows:    646,098
Reduction:           13.3x


## 8. Map Technology Codes to Names

In [9]:
### Map technology codes to human-readable names
block_level['technology_name'] = block_level['technology'].map(TECHNOLOGY_MAP)

### Check for any unmapped codes (would result in NaN)
unmapped_rows = block_level[block_level['technology_name'].isna()]
if unmapped_rows.shape[0] > 0:
    print(f"WARNING: {unmapped_rows.shape[0]} rows with unmapped technology codes")
    print(unmapped_rows['technology'].unique())
else:
    print("All technology codes mapped successfully.")

print("\n--- Technology distribution ---")
print(block_level['technology_name'].value_counts())

All technology codes mapped successfully.

--- Technology distribution ---
technology_name
Cable                           256355
Copper Wire (DSL)               200677
Fiber to the Premises (FTTP)    189066
Name: count, dtype: int64


## 9. Reorder Columns to Match Database Schema

Target schema from the plan:
```
block_geoid, state_fips, county_fips, tract_fips,
provider_id, holding_company, technology, technology_name,
max_download, max_upload, serves_residential, serves_business, low_latency
```

In [10]:
### Reorder columns to match the PostgreSQL block_availability table schema
block_level = block_level[[
    'block_geoid',
    'state_fips',
    'county_fips',
    'tract_fips',
    'provider_id',
    'holding_company',
    'technology',
    'technology_name',
    'max_download',
    'max_upload',
    'serves_residential',
    'serves_business',
    'low_latency'
]]

print(f"Final columns: {list(block_level.columns)}")
print(f"Final shape: {block_level.shape}")
block_level.head(10)

Final columns: ['block_geoid', 'state_fips', 'county_fips', 'tract_fips', 'provider_id', 'holding_company', 'technology', 'technology_name', 'max_download', 'max_upload', 'serves_residential', 'serves_business', 'low_latency']
Final shape: (646098, 13)


,block_geoid,state_fips,county_fips,tract_fips,provider_id,holding_company,technology,technology_name,max_download,max_upload,serves_residential,serves_business,low_latency
0,170010001001000,17,17001,17001000100,130012,Adams Telephone Co-Operative,50,Fiber to the Premises (FTTP),1000,1000,True,True,1
1,170010001001000,17,17001,17001000100,130317,Comcast Corporation,40,Cable,1200,35,True,True,1
2,170010001001001,17,17001,17001000100,130012,Adams Telephone Co-Operative,50,Fiber to the Premises (FTTP),1000,1000,True,True,1
3,170010001001001,17,17001,17001000100,130317,Comcast Corporation,40,Cable,1200,35,True,True,1
4,170010001001003,17,17001,17001000100,130012,Adams Telephone Co-Operative,50,Fiber to the Premises (FTTP),1000,1000,True,True,1
5,170010001001003,17,17001,17001000100,130317,Comcast Corporation,40,Cable,1200,35,True,True,1
6,170010001001005,17,17001,17001000100,130012,Adams Telephone Co-Operative,50,Fiber to the Premises (FTTP),1000,1000,True,True,1
7,170010001001005,17,17001,17001000100,130317,Comcast Corporation,40,Cable,1200,35,True,True,1
8,170010001001007,17,17001,17001000100,130012,Adams Telephone Co-Operative,50,Fiber to the Premises (FTTP),1000,1000,True,True,1
9,170010001001007,17,17001,17001000100,130317,Comcast Corporation,40,Cable,1200,35,True,True,1


## 10. Validation (Step 1.4)

Sanity checks before export:
- No null `block_geoid`
- All `block_geoid` are 15 digits
- All technology codes are mapped
- Speed values are reasonable
- Residential/business flags are correct

In [11]:
### Validation checks
print("=== VALIDATION REPORT ===")
print()

# 1. Null block_geoid
null_geoid = block_level['block_geoid'].isna().sum()
print(f"[{'PASS' if null_geoid == 0 else 'FAIL'}] Null block_geoid: {null_geoid}")

# 2. block_geoid length
bad_len = (block_level['block_geoid'].str.len() != 15).sum()
print(f"[{'PASS' if bad_len == 0 else 'FAIL'}] Invalid block_geoid length: {bad_len}")

# 3. Technology mapping
null_tech_name = block_level['technology_name'].isna().sum()
print(f"[{'PASS' if null_tech_name == 0 else 'FAIL'}] Unmapped technology codes: {null_tech_name}")

# 4. Speed ranges
print(f"\n--- Speed ranges ---")
print(f"Download: {block_level['max_download'].min()} - {block_level['max_download'].max()} Mbps")
print(f"Upload:   {block_level['max_upload'].min()} - {block_level['max_upload'].max()} Mbps")

# 5. Residential/business flag distribution
print(f"\n--- Service type flags ---")
print(f"Serves residential: {block_level['serves_residential'].sum():,} rows")
print(f"Serves business:    {block_level['serves_business'].sum():,} rows")
print(f"Both:               {(block_level['serves_residential'] & block_level['serves_business']).sum():,} rows")

# 6. Summary stats
print(f"\n--- Summary ---")
print(f"Total block-level rows:  {block_level.shape[0]:,}")
print(f"Unique census blocks:    {block_level['block_geoid'].nunique():,}")
print(f"Unique providers:        {block_level['provider_id'].nunique()}")
print(f"Unique holding companies: {block_level['holding_company'].nunique()}")
print(f"Unique counties:         {block_level['county_fips'].nunique()}")
print(f"Unique tracts:           {block_level['tract_fips'].nunique()}")

=== VALIDATION REPORT ===

[PASS] Null block_geoid: 0
[PASS] Invalid block_geoid length: 0
[PASS] Unmapped technology codes: 0

--- Speed ranges ---
Download: 0 - 1200000 Mbps
Upload:   0 - 1200000 Mbps

--- Service type flags ---
Serves residential: 618,006 rows
Serves business:    559,131 rows
Both:               531,039 rows

--- Summary ---
Total block-level rows:  646,098
Unique census blocks:    273,943
Unique providers:        96
Unique holding companies: 96
Unique counties:         102
Unique tracts:           3262


In [12]:
### Sample: pick a census block and inspect its providers/speeds
sample_block = block_level['block_geoid'].value_counts().head(1).index[0]
print(f"Sample census block: {sample_block}")
print(f"(Block with the most provider-technology combinations)\n")

sample = block_level[block_level['block_geoid'] == sample_block][[
    'holding_company', 'technology_name',
    'max_download', 'max_upload', 'serves_residential', 'serves_business', 'low_latency'
]].sort_values('max_download', ascending=False)

print(sample.to_string(index=False))

Sample census block: 170318391002001
(Block with the most provider-technology combinations)

                holding_company              technology_name  max_download  max_upload  serves_residential  serves_business  low_latency
                Zayo Group, LLC Fiber to the Premises (FTTP)         10000       10000               False             True            1
             Pavlov Media, Inc. Fiber to the Premises (FTTP)          2000        2000               False             True            1
            Comcast Corporation                        Cable          2000         250                True             True            1
           Radiate Holdings, LP Fiber to the Premises (FTTP)          1500        1500                True             True            1
           Radiate Holdings, LP                        Cable          1500          50                True             True            1
            Comcast Corporation Fiber to the Premises (FTTP)          1000        100

## 11. Export for Database Loading

In [13]:
### Save the block-level rollup to CSV
output_path = '../output/IL_block_level_availability.csv'
block_level.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Rows: {block_level.shape[0]:,}")
print(f"Columns: {block_level.shape[1]}")

### File size
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"File size: {file_size_mb:.1f} MB")

Saved to: ../output/IL_block_level_availability.csv
Rows: 646,098
Columns: 13
File size: 64.9 MB
